This is the code for the final project in ECSE 343: Numerical Methods for Engs. Our team, group 20, is made up of
- Serge Al Laham
- Emily Wang
- Jeremias Zimmerman
- Steven Thao

# Part 1: Data generation

## 1.2 Data Pre-processing (FFT)

In [130]:
import numpy as np
import pandas as pd

def calibrate(reference_filename: str) -> dict:
    """
    Run once on a representative simulation to fix the frequency bin indices.
    Returns a dict of locked indices used by get_feature_vector for every simulation.
    """
    data = pd.read_csv(f'../data/measurements.csv', header=None)
    signals = np.array([data[col] for col in data.columns]) # (4, 500)
    features = extract_features(signals) # (4, 251)

    indices = {}

    for idx, row in enumerate(features):
        if idx < 2:
            top_idx = np.argmax(np.abs(row[1:])) + 1
            indices[idx] = top_idx

        else:
            ac_mag  = np.abs(row[1:])
            top5_idx = np.argsort(ac_mag)[::-1][:5]
            indices[idx] = top5_idx

    return indices

# reference indices
indices = calibrate('../data/measurements.csv')

def get_feature_vector(data_filename: str, indices: dict) -> np.ndarray:
    """
    Build the feature vector using the fixed bin indices from calibrate().
    Layout: [mag_11, ph_11, mag_21, ph_21,
             DC_3, mag_31, ph_31, ..., ph_35,
             DC_4, mag_41, ph_41, ..., ph_45] -> shape (26,)
    """
    data = pd.read_csv(f'../data/{data_filename}', header=None)
    signals = np.array([data[col] for col in data.columns]) # (4, 500)
    features = extract_features(signals) # (4, 251)

    feature_vector = []

    for idx, row in enumerate(features):
        if idx < 2:
            i = indices[idx]
            feature_vector += [np.abs(row[i]), np.angle(row[i])]

        else:
            dc = np.abs(row[0])
            ac_mag = np.abs(row[1:])
            ac_phs = np.angle(row[1:])

            feature_vector.append(dc)
            for i in indices[idx]:
                feature_vector += [ac_mag[i], ac_phs[i]]

    return np.array(feature_vector)

feature_vector = get_feature_vector('../data/measurements.csv', indices)

# Part 2: Newton-Raphson method

# Part 3: ML solution

In [45]:
# imports
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

In [46]:
"""
    Base class for multivariate polynomial regression
"""
class PolyReg(nn.Module):
    def __init__(self, degree: int, lr: float = 0.001):
        super().__init__()
        self.degree = degree
        self.hidden = nn.Linear(26 * degree, 2)
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        poly_features = [x ** i for i in range(1, self.degree + 1)]
        poly_features = torch.cat(poly_features, dim=0)  # 1D samples → dim=0
        return self.hidden(poly_features)

    """
    Trains the model.

    Input:
        x_train: N x 26 matrix of all training inputs
        y_train: N x 2 matrix of all training labels
        epochs: Training epochs
    Output:
        None
    """
    def train(self, x_train: torch.Tensor, y_train: torch.Tensor, epochs: int = 100) -> None:
        for epoch in range(epochs):
            for x, y in zip(x_train, y_train):
                self.optimizer.zero_grad()
                outputs = self.forward(x)
                loss = self.loss_fn(outputs, y)
                loss.backward()
                self.optimizer.step()
            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    """
        Obtains the test score for the model.

        Input:
            x_test: N x 26 matrix of all test inputs
            y_test: N x 2 matrix of all test labels
        Output:
            resul: validation accuracy for the current test set
    """
    def val_test(self, x_test: torch.Tensor, y_test: torch.Tensor) -> float:
        total_loss = 0.0

        with torch.no_grad():
            for x, y in zip(x_test, y_test):
                pred = self.forward(x)
                loss = self.loss_fn(pred, y)
                total_loss += loss.item()

        return total_loss / len(x_test)

In [47]:
"""
    Class for simple feedforward neural network.
"""
class NeuralNet(nn.Module):
    """
    For now, the activation function is ReLU. Can adapt to other functions in the future.

    Input:
        units: list of number of units in each layer, including input and output layers.
    Output:
        NeuralNet object.
    """
    def __init__(self, units: list[int], lr: float = 0.001) -> None:
        super().__init__()
        self.hidden = nn.ModuleList([nn.Linear(units[i], units[i + 1]) for i in range(len(units) - 1)])
        self.activation = nn.ReLU()
        self.optimizer = optim.Adam(self.parameters(), lr=lr)
        self.loss_fn = nn.MSELoss()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.hidden[:-1]:
            x = self.activation(layer(x))
        return self.hidden[-1](x)

    """
    Trains the model.

    Input:
        x_train: N x 26 matrix of all training inputs
        y_train: N x 2 matrix of all training labels
        epochs: Training epochs
    Output:
        None
    """
    def train(self, x_train: torch.Tensor, y_train: torch.Tensor, epochs: int = 100) -> None:
        for epoch in range(epochs):
            for x, y in zip(x_train, y_train):
                self.optimizer.zero_grad()
                outputs = self.forward(x)
                loss = self.loss_fn(outputs.squeeze(), y)
                loss.backward()
                self.optimizer.step()
            if epoch % 10 == 0:
                print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

    """
        Obtains the test score for the model.

        Input:
            x_test: N x 26 matrix of all test inputs
            y_test: N x 2 matrix of all test labels
        Output:
            resul: validation accuracy for the current test set
    """
    def val_test(self, x_test: torch.Tensor, y_test: torch.Tensor) -> float:
        total_loss = 0.0

        with torch.no_grad():
            for x, y in zip(x_test, y_test):
                pred = self.forward(x)
                loss = self.loss_fn(pred, y)
                total_loss += loss.item()

        return total_loss / len(x_test)

In [48]:
def split_dataset(dataset: tuple[torch.Tensor, torch.Tensor], train_ratio: float = 0.7, val_ratio: float = 0.15) -> tuple[tuple[torch.Tensor, torch.Tensor], tuple[torch.Tensor, torch.Tensor], tuple[torch.Tensor, torch.Tensor]]:
    """
    Split dataset into training, validation, and test sets.

    Input:
        dataset: single tuple of (X, y) where X is all the input data and y is all the labels.
        train_ratio: decimal ratio of training data.
        val_ratio: decimal ratio of validation data.
    Output:
        tuple of (X_train, y_train), (X_val, y_val), (X_test, y_test) where X and y are torch tensors.
    """
    X, y = dataset
    n_samples = X.shape[0]
    # shuffle indices
    indices = np.random.permutation(n_samples)
    # calculate split indices
    train_end = int(train_ratio * n_samples)
    val_end = int((train_ratio + val_ratio) * n_samples)

    # do the split
    X_train, y_train = X[indices[:train_end]], y[indices[:train_end]]
    X_val, y_val = X[indices[train_end:val_end]], y[indices[train_end:val_end]]
    X_test, y_test = X[indices[val_end:]], y[indices[val_end:]]

    return (torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32)), (torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32)), (torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.float32))

In [44]:
# TESTING: Assumed dataset:
# dataset = {(x^(i), y^(i))} for i in [1, N]
# In reality, dataset = (X, y), with X being an N x 26 matrix, and y being an N x 2 matrix.
# y^{(i)} = (R^{(i)}, C^{(i)})
# x^{(i)} = 26-length real-valued vector of features for the ith simulation.

# 1. Split dataset into training, validation, and test sets.
# example dataset with 100 samples, 26 features, and 2 labels
dataset = (np.random.rand(100, 26), np.random.rand(100, 2))
D_train, D_val, D_test = split_dataset(dataset, train_ratio=0.7, val_ratio=0.15)
print("Training set:", D_train[0].shape, D_train[1].shape)
print("Validation set:", D_val[0].shape, D_val[1].shape)
print("Test set:", D_test[0].shape, D_test[1].shape)

# 2. Initialize neural network and polynomial regression model.
units = [26, 64, 32, 2] # 26 inputs, 2 hidden layers, 2 outputs
network = NeuralNet(units, lr=0.001)
poly_reg = PolyReg(degree=3, lr=0.001)
print(network)
print(poly_reg)

# 3. Train neural network on training set, using validation set for early stopping.
network.train(D_train[0], D_train[1], epochs=100)
val_loss = network.val_test(D_val[0], D_val[1])
print(f"Validation Loss: {val_loss:.4f}")

poly_reg.train(D_train[0], D_train[1], epochs=100)
val_loss_poly = poly_reg.val_test(D_val[0], D_val[1])
print(f"Validation Loss (Polynomial Regression): {val_loss_poly:.4f}")

Training set: torch.Size([70, 26]) torch.Size([70, 2])
Validation set: torch.Size([15, 26]) torch.Size([15, 2])
Test set: torch.Size([15, 26]) torch.Size([15, 2])
NeuralNet(
  (hidden): ModuleList(
    (0): Linear(in_features=26, out_features=64, bias=True)
    (1): Linear(in_features=64, out_features=32, bias=True)
    (2): Linear(in_features=32, out_features=2, bias=True)
  )
  (activation): ReLU()
  (loss_fn): MSELoss()
)
PolyReg(
  (hidden): Linear(in_features=78, out_features=2, bias=True)
  (loss_fn): MSELoss()
)
Epoch 0, Loss: 0.0960
Epoch 10, Loss: 0.0478
Epoch 20, Loss: 0.0004
Epoch 30, Loss: 0.0004
Epoch 40, Loss: 0.0079
Epoch 50, Loss: 0.0118
Epoch 60, Loss: 0.0012
Epoch 70, Loss: 0.0012
Epoch 80, Loss: 0.0022
Epoch 90, Loss: 0.0002
Validation Loss: 0.1042
Epoch 0, Loss: 0.1452
Epoch 10, Loss: 0.0572
Epoch 20, Loss: 0.0536
Epoch 30, Loss: 0.0478
Epoch 40, Loss: 0.0429
Epoch 50, Loss: 0.0394
Epoch 60, Loss: 0.0368
Epoch 70, Loss: 0.0349
Epoch 80, Loss: 0.0333
Epoch 90, Loss: 